In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd

# Define paths
xpt_folder = "/content/drive/MyDrive/MSML610 Project/Data"
csv_folder = "/content/drive/MyDrive/MSML610 Project/CSV"

# Create output folder if it doesn't exist
os.makedirs(csv_folder, exist_ok=True)

# Loop through all .xpt files
for file in os.listdir(xpt_folder):
    if file.lower().endswith(".xpt"):
        xpt_path = os.path.join(xpt_folder, file)
        csv_name = os.path.splitext(file)[0] + ".csv"
        csv_path = os.path.join(csv_folder, csv_name)

        try:
            # Read .xpt file
            df = pd.read_sas(xpt_path)
            # Save as CSV
            df.to_csv(csv_path, index=False)
            print(f"✅ Converted: {file} → {csv_name}")
        except Exception as e:
            print(f"❌ Error converting {file}: {e}")

print("\nAll conversions complete! CSV files are saved in:", csv_folder)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Converted: DEMO_L.xpt → DEMO_L.csv
✅ Converted: DR1IFF_L.xpt → DR1IFF_L.csv
✅ Converted: DR2IFF_L.xpt → DR2IFF_L.csv
✅ Converted: DSQIDS_L.xpt → DSQIDS_L.csv
✅ Converted: DSQTOT_L.xpt → DSQTOT_L.csv
✅ Converted: BPXO_L.xpt → BPXO_L.csv
✅ Converted: BMX_L.xpt → BMX_L.csv
✅ Converted: TCHOL_L.xpt → TCHOL_L.csv
✅ Converted: HDL_L.xpt → HDL_L.csv
✅ Converted: TRIGLY_L.xpt → TRIGLY_L.csv
✅ Converted: GHB_L.xpt → GHB_L.csv
✅ Converted: GLU_L.xpt → GLU_L.csv
✅ Converted: HSCRP_L.xpt → HSCRP_L.csv
✅ Converted: RXQ_RX_L.xpt → RXQ_RX_L.csv
✅ Converted: BPQ_L.xpt → BPQ_L.csv
✅ Converted: MCQ_L.xpt → MCQ_L.csv
✅ Converted: DIQ_L.xpt → DIQ_L.csv
✅ Converted: SMQ_L.xpt → SMQ_L.csv
✅ Converted: PAQ_L.xpt → PAQ_L.csv
✅ Converted: ALQ_L.xpt → ALQ_L.csv
✅ Converted: INQ_L.xpt → INQ_L.csv

All conversions complete! CSV files are saved in: /content/drive/MyDrive/MSML610 Projec

In [8]:
import os
import numpy as np
import pandas as pd

# ---------- Paths (adjust if needed) ----------
in_csv  = "/content/drive/MyDrive/MSML610 Project/CSV/DEMO_L.csv"
out_dir = "/content/drive/MyDrive/MSML610 Project/CSV_meaningful"
os.makedirs(out_dir, exist_ok=True)
renamed_csv = os.path.join(out_dir, "DEMO_L_meaningful.csv")
labeled_csv = os.path.join(out_dir, "DEMO_L_meaningful_labeled.csv")

# ---------- Map: NHANES code -> meaningful column name ----------
CODE_TO_NAME = {
    "SEQN":      "respondent_id",
    "SDDSRVYR":  "release_cycle",
    "RIDSTATR":  "interview_exam_status",
    "RIAGENDR":  "gender",
    "RIDAGEYR":  "age_years_screening",
    "RIDAGEMN":  "age_months_screening_0_24",
    "RIDRETH1":  "race_ethnicity",
    "RIDRETH3":  "race_ethnicity_with_nh_asian",
    "RIDEXMON":  "exam_six_month_period",
    "RIDEXAGM":  "age_months_exam_0_19",
    "DMQMILIZ":  "served_us_armed_forces",
    "DMDBORN4":  "country_of_birth",
    "DMDYRUSR":  "years_in_us",
    "DMDEDUC2":  "education_level_adult_20plus",
    "DMDMARTZ":  "marital_status",
    "RIDEXPRG":  "pregnancy_status_exam",
    "DMDHHSIZ":  "household_size",
    "DMDHRGND":  "hh_ref_gender",
    "DMDHRAGZ":  "hh_ref_age_years",
    "DMDHREDZ":  "hh_ref_education_level",
    "DMDHRMAZ":  "hh_ref_marital_status",
    "DMDHSEDZ":  "hh_spouse_education_level",
    "WTINT2YR":  "wt_interview_2yr",
    "WTMEC2YR":  "wt_mec_exam_2yr",
    "SDMVSTRA":  "var_pseudostratum",
    "SDMVPSU":   "var_pseudo_psu",
    "INDFMPIR":  "income_to_poverty_ratio",
}

# ---------- Label dictionaries (NHANES codes) ----------
LBL = {}

LBL["release_cycle"] = {12: "NHANES Aug 2021–Aug 2023 public release"}

LBL["interview_exam_status"] = {
    1: "Interviewed only",
    2: "Both interviewed and MEC examined",
}

LBL["gender"] = {1: "Male", 2: "Female"}

# race/ethnicity
LBL["race_ethnicity"] = {
    1: "Mexican American",
    2: "Other Hispanic",
    3: "Non-Hispanic White",
    4: "Non-Hispanic Black",
    5: "Other Race - Including Multi-Racial",
}
LBL["race_ethnicity_with_nh_asian"] = {
    1: "Mexican American",
    2: "Other Hispanic",
    3: "Non-Hispanic White",
    4: "Non-Hispanic Black",
    6: "Non-Hispanic Asian",
    7: "Other Race - Including Multi-Racial",
}

# exam six-month period (full phrasing)
LBL["exam_six_month_period"] = {
    1: "November 1–April 30",
    2: "May 1–October 31",
}

# military service
LBL["served_us_armed_forces"] = {
    1: "Yes",
    2: "No",
    7: "Refused",
    9: "Don't know",
}

# country of birth (include DC)
LBL["country_of_birth"] = {
    1: "Born in 50 US states or Washington, DC",
    2: "Others",
    77: "Refused",
    99: "Don't know",
}

# years in US
LBL["years_in_us"] = {
    1: "Less than 1 year",
    2: "1 to 4 years",
    3: "5 to 9 years",
    4: "10 to 14 years",
    5: "15 to 19 years",
    6: "20 years or more",
    77: "Refused",
    99: "Don't know",
}

# education 20+
LBL["education_level_adult_20plus"] = {
    1: "Less than 9th grade",
    2: "9–11th grade/no diploma",
    3: "High school graduate/GED",
    4: "Some college or AA degree",
    5: "College graduate or above",
    7: "Refused",
    9: "Don't know",
}

# marital status
LBL["marital_status"] = {
    1: "Married/Living with partner",
    2: "Widowed/Divorced/Separated",
    3: "Never married",
    77: "Refused",
    99: "Don't know",
}

# pregnancy
LBL["pregnancy_status_exam"] = {
    1: "Pregnant (positive test or self-reported)",
    2: "Not pregnant at exam",
    3: "Cannot ascertain pregnancy at exam",
}

# household ref attributes (for sampled persons 0–19y)
LBL["hh_ref_gender"] = {1: "Male", 2: "Female"}
LBL["hh_ref_age_years"] = {
    1: "<20 years",
    2: "20–39 years",
    3: "40–59 years",
    4: "60+ years",
}
LBL["hh_ref_education_level"] = {
    1: "Less than high school degree",
    2: "HS grad/GED or some college/AA degree",
    3: "College graduate or above",
}
LBL["hh_ref_marital_status"] = {
    1: "Married/Living with partner",
    2: "Widowed/Divorced/Separated",
    3: "Never Married",
}
LBL["hh_spouse_education_level"] = {
    1: "Less than high school degree",
    2: "HS grad/GED or some college/AA degree",
    3: "College graduate or above",
}

CATEGORICALS = [
    "release_cycle",
    "interview_exam_status",
    "gender",
    "race_ethnicity",
    "race_ethnicity_with_nh_asian",
    "exam_six_month_period",
    "served_us_armed_forces",
    "country_of_birth",
    "years_in_us",
    "education_level_adult_20plus",
    "marital_status",
    "pregnancy_status_exam",
    "hh_ref_gender",
    "hh_ref_age_years",
    "hh_ref_education_level",
    "hh_ref_marital_status",
    "hh_spouse_education_level",
]

# ---------- Helpers ----------
def with_counts(series: pd.Series, base_map: dict, missing_label: str = "Missing") -> pd.Series:
    """
    Map codes -> 'Label [count]' using counts computed on the series.
    - Coerces numeric-looking strings to numeric so integer keys match.
    - Unknown non-numeric codes fall back to 'raw_value [count]' (not NaN).
    - True missing becomes 'Missing [count]'.
    Returns a string Series.
    """
    # Coerce numeric-like values so mapping dict keys (ints) match
    s_num = pd.to_numeric(series, errors="coerce")
    vc_num = s_num.value_counts(dropna=False)

    # Build known code labels with counts
    code_to_label_count = {
        code: f"{label} [{int(vc_num.get(code, 0))}]"
        for code, label in base_map.items()
    }

    # Start with mapped numeric codes
    out = s_num.map(code_to_label_count)

    # Handle entries that didn't coerce to numeric but are not missing: keep raw value with its count
    mask_non_numeric = s_num.isna() & series.notna()
    if mask_non_numeric.any():
        raw = series[mask_non_numeric]
        raw_vc = raw.value_counts(dropna=False)
        fallback_map = {k: f"{k} [{int(raw_vc.get(k, 0))}]" for k in raw_vc.index}
        out.loc[mask_non_numeric] = raw.map(fallback_map)

    # True missing
    miss_ct = int(vc_num.get(np.nan, 0))
    out = out.fillna(f"{missing_label} [{miss_ct}]")

    return out.astype("string")

# ---------- 1) Read original, rename, write ----------
df = pd.read_csv(in_csv)

present_to_name = {col: CODE_TO_NAME[col] for col in df.columns if col in CODE_TO_NAME}
missing = [k for k in CODE_TO_NAME if k not in df.columns]
if missing:
    print("Note: codebook fields not found in the CSV header, not renamed:")
    print(", ".join(missing))

df = df.rename(columns=present_to_name)
df.to_csv(renamed_csv, index=False)
print(f"Saved renamed CSV to: {renamed_csv}")

# ---------- 2) Read renamed, clean/label, write ----------
df = pd.read_csv(renamed_csv)

# Fix microscopic weights (rare XPT->CSV artifact)
if "wt_mec_exam_2yr" in df.columns:
    wt = pd.to_numeric(df["wt_mec_exam_2yr"], errors="coerce")
    df["wt_mec_exam_2yr"] = np.where((wt.notna()) & (np.abs(wt) < 1e-8), 0.0, wt)

# Apply categorical mappings with counts
for col in CATEGORICALS:
    if col in df.columns:
        df[col] = with_counts(df[col], LBL[col], missing_label="Missing")

# Special case: label Not MEC Examined for zeros in wt_mec_exam_2yr
if "wt_mec_exam_2yr" in df.columns:
    wt = df["wt_mec_exam_2yr"]
    # After earlier step, wt is numeric except where truly NaN
    is_zero = pd.to_numeric(wt, errors="coerce").fillna(0.0).eq(0.0)
    zero_ct = int(is_zero.sum())
    not_exam_label = f"Not MEC Examined [{zero_ct}]"
    df["wt_mec_exam_2yr"] = np.where(is_zero, not_exam_label, wt)

# Leave continuous/range fields numeric:
#   age_years_screening, age_months_screening_0_24, age_months_exam_0_19,
#   household_size, wt_interview_2yr, var_pseudostratum, var_pseudo_psu,
#   income_to_poverty_ratio
# (WTMEC2YR handled above.)

# ---------- Optional sanity checks (print warnings, don't crash) ----------
try:
    if {"interview_exam_status", "exam_six_month_period"}.issubset(df.columns):
        interviewed_only = df["interview_exam_status"].str.contains("Interviewed only")
        exam_missing = df["exam_six_month_period"].str.contains("Missing")
        if interviewed_only.sum() != exam_missing.sum():
            print("[WARN] RIDSTATR 'Interviewed only' count != RIDEXMON missing count.")
    if {"interview_exam_status", "wt_mec_exam_2yr"}.issubset(df.columns):
        interviewed_only = df["interview_exam_status"].str.contains("Interviewed only")
        not_exam = df["wt_mec_exam_2yr"].astype(str).str.startswith("Not MEC Examined [")
        if interviewed_only.sum() != not_exam.sum():
            print("[WARN] RIDSTATR 'Interviewed only' count != WTMEC2YR zeros.")
except Exception as e:
    print(f"[WARN] Sanity checks skipped: {e}")

# ---------- Save ----------
df.to_csv(labeled_csv, index=False)
print(f"Saved labeled CSV to: {labeled_csv}")

Saved renamed CSV to: /content/drive/MyDrive/MSML610 Project/CSV_meaningful/DEMO_L_meaningful.csv
Saved labeled CSV to: /content/drive/MyDrive/MSML610 Project/CSV_meaningful/DEMO_L_meaningful_labeled.csv


In [12]:
import os
import numpy as np
import pandas as pd

# ---------- Paths ----------
in_csv  = "/content/drive/MyDrive/MSML610 Project/CSV/DR1IFF_L.csv"
out_dir = "/content/drive/MyDrive/MSML610 Project/CSV_meaningful"
os.makedirs(out_dir, exist_ok=True)
renamed_csv = os.path.join(out_dir, "DR1IFF_L_meaningful.csv")
labeled_csv = os.path.join(out_dir, "DR1IFF_L_meaningful_labeled.csv")

# ---------- Read with "." treated as missing ----------
df = pd.read_csv(in_csv, na_values=["."], keep_default_na=True)

# ---------- Map: NHANES code -> meaningful column name ----------
CODE_TO_NAME = {
    "SEQN":      "respondent_id",
    "WTDRD1":    "wt_diet_day1",
    "WTDR2D":    "wt_diet_day2",
    "DR1ILINE":  "food_item_line_number",
    "DR1DRSTZ":  "dietary_recall_status",
    "DR1EXMER":  "interviewer_id_code",
    "DRABF":     "breast_fed_infant_either_day",
    "DRDINT":    "num_days_of_intake",
    "DR1DBIH":   "days_between_intake_and_hh_interview",
    "DR1DAY":    "intake_day_of_week",
    "DR1LANG":   "respondent_language",
    "DR1CCMNM":  "combination_food_number",
    "DR1CCMTX":  "combination_food_type",
    "DR1_020":   "time_of_eating_occasion_hhmm",
    "DR1_030Z":  "eating_occasion_name",
    "DR1FS":     "food_source",
    "DR1_040Z":  "ate_meal_at_home",
    "DR1IFDCD":  "usda_food_code",
    "DR1IGRMS":  "grams",
    "DR1IKCAL":  "energy_kcal",
    "DR1IPROT":  "protein_g",
    "DR1ICARB":  "carbohydrate_g",
    "DR1ISUGR":  "sugars_total_g",
    "DR1IFIBE":  "fiber_dietary_g",
    "DR1ITFAT":  "fat_total_g",
    "DR1ISFAT":  "fat_saturated_g",
    "DR1IMFAT":  "fat_monounsaturated_g",
    "DR1IPFAT":  "fat_polyunsaturated_g",
    "DR1ICHOL":  "cholesterol_mg",
    "DR1IATOC":  "vitamin_e_alpha_tocopherol_mg",
    "DR1IATOA":  "vitamin_e_alpha_tocopherol_added_mg",
    "DR1IRET":   "retinol_mcg",
    "DR1IVARA":  "vitamin_a_rae_mcg",
    "DR1IACAR":  "alpha_carotene_mcg",
    "DR1IBCAR":  "beta_carotene_mcg",
    "DR1ICRYP":  "beta_cryptoxanthin_mcg",
    "DR1ILYCO":  "lycopene_mcg",
    "DR1ILZ":    "lutein_zeaxanthin_mcg",
    "DR1IVB1":   "thiamin_b1_mg",
    "DR1IVB2":   "riboflavin_b2_mg",
    "DR1INIAC":  "niacin_mg",
    "DR1IVB6":   "vitamin_b6_mg",
    "DR1IFOLA":  "folate_total_mcg",
    "DR1IFA":    "folic_acid_mcg",
    "DR1IFF":    "food_folate_mcg",
    "DR1IFDFE":  "folate_dfe_mcg",
    "DR1ICHL":   "choline_total_mg",
    "DR1IVB12":  "vitamin_b12_mcg",
    "DR1IB12A":  "vitamin_b12_added_mcg",
    "DR1IVC":    "vitamin_c_mg",
    "DR1IVD":    "vitamin_d_d2_d3_mcg",
    "DR1IVK":    "vitamin_k_mcg",
    "DR1ICALC":  "calcium_mg",
    "DR1IPHOS":  "phosphorus_mg",
    "DR1IMAGN":  "magnesium_mg",
    "DR1IIRON":  "iron_mg",
    "DR1IZINC":  "zinc_mg",
    "DR1ICOPP":  "copper_mg",
    "DR1ISODI":  "sodium_mg",
    "DR1IPOTA":  "potassium_mg",
    "DR1ISELE":  "selenium_mcg",
    "DR1ICAFF":  "caffeine_mg",
    "DR1ITHEO":  "theobromine_mg",
    "DR1IALCO":  "alcohol_g",
    "DR1IMOIS":  "moisture_g",
    "DR1IS040":  "sfa_4_0_butanoic_g",
    "DR1IS060":  "sfa_6_0_hexanoic_g",
    "DR1IS080":  "sfa_8_0_octanoic_g",
    "DR1IS100":  "sfa_10_0_decanoic_g",
    "DR1IS120":  "sfa_12_0_dodecanoic_g",
    "DR1IS140":  "sfa_14_0_tetradecanoic_g",
    "DR1IS160":  "sfa_16_0_hexadecanoic_g",
    "DR1IS180":  "sfa_18_0_octadecanoic_g",
    "DR1IM161":  "mfa_16_1_hexadecenoic_g",
    "DR1IM181":  "mfa_18_1_octadecenoic_g",
    "DR1IM201":  "mfa_20_1_eicosenoic_g",
    "DR1IM221":  "mfa_22_1_docosenoic_g",
    "DR1IP182":  "pfa_18_2_octadecadienoic_g",
    "DR1IP183":  "pfa_18_3_octadecatrienoic_g",
    "DR1IP184":  "pfa_18_4_octadecatetraenoic_g",
    "DR1IP204":  "pfa_20_4_eicosatetraenoic_g",
    "DR1IP205":  "pfa_20_5_eicosapentaenoic_g",
    "DR1IP225":  "pfa_22_5_docosapentaenoic_g",
    "DR1IP226":  "pfa_22_6_docosahexaenoic_g",
}

# Rename
present_to_name = {c: CODE_TO_NAME[c] for c in df.columns if c in CODE_TO_NAME}
missing = [k for k in CODE_TO_NAME if k not in df.columns]
if missing:
    print("Note: not found in header (not renamed):", ", ".join(missing))
df = df.rename(columns=present_to_name)

# Save renamed (clean headers, "." already treated as NaN)
df.to_csv(renamed_csv, index=False)
print(f"Saved renamed CSV to: {renamed_csv}")

# ---------- Label dictionaries for selected categorical vars ----------
LBL = {
    "dietary_recall_status": {
        1: "Reliable and met the minimum criteria",
        2: "Not reliable or not met the minimum criteria",
        4: "Reported consuming breast-milk",
        5: "Not done",
    },
    "breast_fed_infant_either_day": {
        1: "Yes",
        2: "No",
    },
    "num_days_of_intake": {
        1: "Day 1 only",
        2: "Day 1 and day 2",
    },
    "intake_day_of_week": {
        1: "Sunday",
        2: "Monday",
        3: "Tuesday",
        4: "Wednesday",
        5: "Thursday",
        6: "Friday",
        7: "Saturday",
    },
    "respondent_language": {
        1: "English",
        2: "Spanish",
        3: "English and Spanish",
        4: "Other",
    },
    "combination_food_type": {
        0: "Non-combination food",
        1: "Beverage w/ additions",
        2: "Cereal w/ additions",
        3: "Bread/baked products w/ additions",
        4: "Salad",
        5: "Sandwiches",
        6: "Soup",
        7: "Frozen meals",
        8: "Ice cream/frozen yogurt w/ additions",
        9: "Dried beans and vegetable w/ additions",
        10: "Fruit w/ additions",
        11: "Tortilla products",
        12: "Meat, poultry, fish",
        13: "Lunchables®",
        14: "Chips w/ additions",
        15: "Baby Toddler Food and Infant Formula",
        90: "Other mixtures",
    },
    "eating_occasion_name": {
        1: "Breakfast", 2: "Lunch", 3: "Dinner", 4: "Supper", 5: "Brunch",
        6: "Snack", 7: "Drink", 8: "Infant feeding", 9: "Extended consumption",
        10: "Desayano", 11: "Almuerzo", 12: "Comida", 13: "Merienda",
        14: "Cena", 15: "Entre comida", 16: "Botana", 17: "Bocadillo",
        18: "Tentempie", 19: "Bebida", 91: "Other", 99: "Don't know",
    },
    "food_source": {
        1:"Store - grocery/supermarket", 2:"Restaurant with waiter/waitress",
        3:"Restaurant fast food/pizza", 4:"Bar/tavern/lounge",
        5:"Restaurant no additional information", 6:"Cafeteria NOT in a K-12 school",
        7:"Cafeteria in a K-12 school", 8:"Child/Adult care center",
        9:"Child/Adult home care", 10:"Soup kitchen/shelter/food pantry",
        11:"Meals on Wheels", 12:"Community food program - other",
        13:"Community program no additional information", 14:"Vending machine",
        15:"Common coffee pot or snack tray", 16:"From someone else/gift",
        17:"Mail order purchase", 18:"Residential dining facility",
        19:"Grown or caught by you or someone you know",
        20:"Fish caught by you or someone you know",
        24:"Sport, recreation, or entertainment facility",
        25:"Street vendor, vending truck", 26:"Fundraiser sales",
        27:"Store - convenience type", 28:"Store - no additional info",
        91:"Other, specify", 99:"Don't know",
    },
    "ate_meal_at_home": {
        1: "Yes",
        2: "No",
        7: "Refused",
        9: "Don't know",
    },
}

# Categorical columns to label (the ones with discrete codebooks above)
CATEGORICALS = [
    "dietary_recall_status",
    "breast_fed_infant_either_day",
    "num_days_of_intake",
    "intake_day_of_week",
    "respondent_language",
    "combination_food_type",
    "eating_occasion_name",
    "food_source",
    "ate_meal_at_home",
]

# ---------- Helper: safe mapper with counts; respects global NaN (incl ".") ----------
def with_counts(series: pd.Series, base_map: dict, missing_label: str = "Missing") -> pd.Series:
    """
    Map codes -> 'Label [count]'. Count is computed on the *coerced* numeric view.
    Unknown non-numeric codes fall back to 'raw_value [count]'.
    True missing (NaN, including '.') becomes 'Missing [count]'.
    """
    s_num = pd.to_numeric(series, errors="coerce")
    vc_num = s_num.value_counts(dropna=False)

    code_to_label_count = {
        code: f"{label} [{int(vc_num.get(code, 0))}]"
        for code, label in base_map.items()
    }

    out = s_num.map(code_to_label_count)

    # Non-numeric but not missing -> keep raw tokens with their counts
    mask_non_numeric = s_num.isna() & series.notna()
    if mask_non_numeric.any():
        raw = series[mask_non_numeric]
        raw_vc = raw.value_counts(dropna=False)
        fallback = {k: f"{k} [{int(raw_vc.get(k, 0))}]" for k in raw_vc.index}
        out.loc[mask_non_numeric] = raw.map(fallback)

    miss_ct = int(vc_num.get(np.nan, 0))
    out = out.fillna(f"{missing_label} [{miss_ct}]")
    return out.astype("string")

# ---------- Write labeled version ----------
df_lab = df.copy()

# Weights: label zeros as "not done/incomplete [count]" but keep positive values numeric
for wt_col, not_done_label in [
    ("wt_diet_day1", "Day 1 dietary recall not done/incomplete"),
    ("wt_diet_day2", "Day 2 dietary recall not done/incomplete"),
]:
    if wt_col in df_lab.columns:
        s = pd.to_numeric(df_lab[wt_col], errors="coerce")
        is_zero = s.fillna(0.0).eq(0.0)
        zero_ct = int(is_zero.sum())
        label = f"{not_done_label} [{zero_ct}]"
        df_lab[wt_col] = np.where(is_zero, label, s)

# Categorical mappings with counts
for col in CATEGORICALS:
    if col in df_lab.columns:
        df_lab[col] = with_counts(df_lab[col], LBL[col], missing_label="Missing")

# Leave these numeric/ID/time fields as numeric (they’re ranges, not coded categories):
NUMERIC_KEEP = [
    "respondent_id",
    "food_item_line_number",
    "interviewer_id_code",                 # ID code, numeric range
    "days_between_intake_and_hh_interview",# -30..149
    "combination_food_number",             # 0..15
    "time_of_eating_occasion_hhmm",        # HHMM6 numeric
    "usda_food_code",
    "grams","energy_kcal","protein_g","carbohydrate_g","sugars_total_g",
    "fiber_dietary_g","fat_total_g","fat_saturated_g","fat_monounsaturated_g",
    "fat_polyunsaturated_g","cholesterol_mg","vitamin_e_alpha_tocopherol_mg",
    "vitamin_e_alpha_tocopherol_added_mg","retinol_mcg","vitamin_a_rae_mcg",
    "alpha_carotene_mcg","beta_carotene_mcg","beta_cryptoxanthin_mcg",
    "lycopene_mcg","lutein_zeaxanthin_mcg","thiamin_b1_mg","riboflavin_b2_mg",
    "niacin_mg","vitamin_b6_mg","folate_total_mcg","folic_acid_mcg",
    "food_folate_mcg","folate_dfe_mcg","choline_total_mg","vitamin_b12_mcg",
    "vitamin_b12_added_mcg","vitamin_c_mg","vitamin_d_d2_d3_mcg","vitamin_k_mcg",
    "calcium_mg","phosphorus_mg","magnesium_mg","iron_mg","zinc_mg","copper_mg",
    "sodium_mg","potassium_mg","selenium_mcg","caffeine_mg","theobromine_mg",
    "alcohol_g","moisture_g","sfa_4_0_butanoic_g","sfa_6_0_hexanoic_g",
    "sfa_8_0_octanoic_g","sfa_10_0_decanoic_g","sfa_12_0_dodecanoic_g",
    "sfa_14_0_tetradecanoic_g","sfa_16_0_hexadecanoic_g","sfa_18_0_octadecanoic_g",
    "mfa_16_1_hexadecenoic_g","mfa_18_1_octadecenoic_g","mfa_20_1_eicosenoic_g",
    "mfa_22_1_docosenoic_g","pfa_18_2_octadecadienoic_g","pfa_18_3_octadecatrienoic_g",
    "pfa_18_4_octadecatetraenoic_g","pfa_20_4_eicosatetraenoic_g",
    "pfa_20_5_eicosapentaenoic_g","pfa_22_5_docosapentaenoic_g","pfa_22_6_docosahexaenoic_g",
]

for col in NUMERIC_KEEP:
    if col in df_lab.columns:
        df_lab[col] = pd.to_numeric(df_lab[col], errors="coerce")

# ---------- Save labeled ----------
df_lab.to_csv(labeled_csv, index=False)
print(f"Saved labeled CSV to: {labeled_csv}")

Saved renamed CSV to: /content/drive/MyDrive/MSML610 Project/CSV_meaningful/DR1IFF_L_meaningful.csv
Saved labeled CSV to: /content/drive/MyDrive/MSML610 Project/CSV_meaningful/DR1IFF_L_meaningful_labeled.csv


In [15]:
import os
import numpy as np
import pandas as pd

# ---------- Paths ----------
in_csv  = "/content/drive/MyDrive/MSML610 Project/CSV/DR2IFF_L.csv"
out_dir = "/content/drive/MyDrive/MSML610 Project/CSV_meaningful"
os.makedirs(out_dir, exist_ok=True)
renamed_csv = os.path.join(out_dir, "DR2IFF_L_meaningful.csv")
labeled_csv = os.path.join(out_dir, "DR2IFF_L_meaningful_labeled.csv")

# ---------- Read with "." treated as missing ----------
df = pd.read_csv(in_csv, na_values=["."], keep_default_na=True)

# ---------- Map: NHANES code -> meaningful snake_case name ----------
CODE_TO_NAME = {
    "SEQN":      "respondent_id",
    "WTDRD1":    "wt_diet_day1",
    "WTDR2D":    "wt_diet_day2",
    "DR2ILINE":  "food_item_line_number",
    "DR2DRSTZ":  "dietary_recall_status",
    "DR2EXMER":  "interviewer_id_code",
    "DRABF":     "breast_fed_infant_either_day",
    "DRDINT":    "num_days_of_intake",
    "DR2DBIH":   "days_between_intake_and_hh_interview",
    "DR2DAY":    "intake_day_of_week",
    "DR2LANG":   "respondent_language",
    "DR2CCMNM":  "combination_food_number",
    "DR2CCMTX":  "combination_food_type",
    "DR2_020":   "time_of_eating_occasion_hhmm",
    "DR2_030Z":  "eating_occasion_name",
    "DR2FS":     "food_source",
    "DR2_040Z":  "ate_meal_at_home",
    "DR2IFDCD":  "usda_food_code",
    "DR2IGRMS":  "grams",
    "DR2IKCAL":  "energy_kcal",
    "DR2IPROT":  "protein_g",
    "DR2ICARB":  "carbohydrate_g",
    "DR2ISUGR":  "sugars_total_g",
    "DR2IFIBE":  "fiber_dietary_g",
    "DR2ITFAT":  "fat_total_g",
    "DR2ISFAT":  "fat_saturated_g",
    "DR2IMFAT":  "fat_monounsaturated_g",
    "DR2IPFAT":  "fat_polyunsaturated_g",
    "DR2ICHOL":  "cholesterol_mg",
    "DR2IATOC":  "vitamin_e_alpha_tocopherol_mg",
    "DR2IATOA":  "vitamin_e_alpha_tocopherol_added_mg",
    "DR2IRET":   "retinol_mcg",
    "DR2IVARA":  "vitamin_a_rae_mcg",
    "DR2IACAR":  "alpha_carotene_mcg",
    "DR2IBCAR":  "beta_carotene_mcg",
    "DR2ICRYP":  "beta_cryptoxanthin_mcg",
    "DR2ILYCO":  "lycopene_mcg",
    "DR2ILZ":    "lutein_zeaxanthin_mcg",
    "DR2IVB1":   "thiamin_b1_mg",
    "DR2IVB2":   "riboflavin_b2_mg",
    "DR2INIAC":  "niacin_mg",
    "DR2IVB6":   "vitamin_b6_mg",
    "DR2IFOLA":  "folate_total_mcg",
    "DR2IFA":    "folic_acid_mcg",
    "DR2IFF":    "food_folate_mcg",
    "DR2IFDFE":  "folate_dfe_mcg",
    "DR2ICHL":   "choline_total_mg",
    "DR2IVB12":  "vitamin_b12_mcg",
    "DR2IB12A":  "vitamin_b12_added_mcg",
    "DR2IVC":    "vitamin_c_mg",
    "DR2IVD":    "vitamin_d_d2_d3_mcg",
    "DR2IVK":    "vitamin_k_mcg",
    "DR2ICALC":  "calcium_mg",
    "DR2IPHOS":  "phosphorus_mg",
    "DR2IMAGN":  "magnesium_mg",
    "DR2IIRON":  "iron_mg",
    "DR2IZINC":  "zinc_mg",
    "DR2ICOPP":  "copper_mg",
    "DR2ISODI":  "sodium_mg",
    "DR2IPOTA":  "potassium_mg",
    "DR2ISELE":  "selenium_mcg",
    "DR2ICAFF":  "caffeine_mg",
    "DR2ITHEO":  "theobromine_mg",
    "DR2IALCO":  "alcohol_g",
    "DR2IMOIS":  "moisture_g",
    "DR2IS040":  "sfa_4_0_butanoic_g",
    "DR2IS060":  "sfa_6_0_hexanoic_g",
    "DR2IS080":  "sfa_8_0_octanoic_g",
    "DR2IS100":  "sfa_10_0_decanoic_g",
    "DR2IS120":  "sfa_12_0_dodecanoic_g",
    "DR2IS140":  "sfa_14_0_tetradecanoic_g",
    "DR2IS160":  "sfa_16_0_hexadecanoic_g",
    "DR2IS180":  "sfa_18_0_octadecanoic_g",
    "DR2IM161":  "mfa_16_1_hexadecenoic_g",
    "DR2IM181":  "mfa_18_1_octadecenoic_g",
    "DR2IM201":  "mfa_20_1_eicosenoic_g",
    "DR2IM221":  "mfa_22_1_docosenoic_g",
    "DR2IP182":  "pfa_18_2_octadecadienoic_g",
    "DR2IP183":  "pfa_18_3_octadecatrienoic_g",
    "DR2IP184":  "pfa_18_4_octadecatetraenoic_g",
    "DR2IP204":  "pfa_20_4_eicosatetraenoic_g",
    "DR2IP205":  "pfa_20_5_eicosapentaenoic_g",
    "DR2IP225":  "pfa_22_5_docosapentaenoic_g",
    "DR2IP226":  "pfa_22_6_docosahexaenoic_g",
}

# Rename
present_to_name = {c: CODE_TO_NAME[c] for c in df.columns if c in CODE_TO_NAME}
missing = [k for k in CODE_TO_NAME if k not in df.columns]
if missing:
    print("Note: not found in header (not renamed):", ", ".join(missing))
df = df.rename(columns=present_to_name)

# Save renamed
df.to_csv(renamed_csv, index=False)
print(f"Saved renamed CSV to: {renamed_csv}")

# ---------- Label dictionaries for listed categorical vars ----------
LBL = {
    "dietary_recall_status": {
        1: "Reliable and met the minimum criteria",
        2: "Not reliable or not met the minimum criteria",
        4: "Reported consuming breast-milk",
        5: "Not done",
    },
    "breast_fed_infant_either_day": {
        1: "Yes",
        2: "No",
    },
    "num_days_of_intake": {
        1: "Day 1 only",
        2: "Day 1 and day 2",
    },
    "intake_day_of_week": {
        1: "Sunday", 2: "Monday", 3: "Tuesday", 4: "Wednesday",
        5: "Thursday", 6: "Friday", 7: "Saturday",
    },
    "respondent_language": {
        1: "English",
        2: "Spanish",
        3: "English and Spanish",
        4: "Other",
    },
    "combination_food_type": {
        0: "Non-combination food",
        1: "Beverage w/ additions",
        2: "Cereal w/ additions",
        3: "Bread/baked products w/ additions",
        4: "Salad",
        5: "Sandwiches",
        6: "Soup",
        7: "Frozen meals",
        8: "Ice cream/frozen yogurt w/ additions",
        9: "Dried beans and vegetable w/ additions",
        10: "Fruit w/ additions",
        11: "Tortilla products",
        12: "Meat, poultry, fish",
        13: "Lunchables®",
        14: "Chips w/ additions",
        15: "Baby Toddler Food and Infant Formula",
        90: "Other mixtures",
    },
    "eating_occasion_name": {
        1: "Breakfast", 2: "Lunch", 3: "Dinner", 4: "Supper", 5: "Brunch",
        6: "Snack", 7: "Drink", 8: "Infant feeding", 9: "Extended consumption",
        10: "Desayano", 11: "Almuerzo", 12: "Comida", 13: "Merienda",
        14: "Cena", 15: "Entre comida", 16: "Botana", 17: "Bocadillo",
        18: "Tentempie", 19: "Bebida", 91: "Other", 99: "Don't know",
    },
    "food_source": {
        1:"Store - grocery/supermarket", 2:"Restaurant with waiter/waitress",
        3:"Restaurant fast food/pizza", 4:"Bar/tavern/lounge",
        5:"Restaurant no additional information", 6:"Cafeteria NOT in a K-12 school",
        7:"Cafeteria in a K-12 school", 8:"Child/Adult care center",
        9:"Child/Adult home care", 10:"Soup kitchen/shelter/food pantry",
        11:"Meals on Wheels", 12:"Community food program - other",
        13:"Community program no additional information", 14:"Vending machine",
        15:"Common coffee pot or snack tray", 16:"From someone else/gift",
        17:"Mail order purchase", 18:"Residential dining facility",
        19:"Grown or caught by you or someone you know",
        20:"Fish caught by you or someone you know",
        24:"Sport, recreation, or entertainment facility",
        25:"Street vendor, vending truck", 26:"Fundraiser sales",
        27:"Store - convenience type", 28:"Store - no additional info",
        91:"Other, specify", 99:"Don't know",
    },
    "ate_meal_at_home": {
        1: "Yes",
        2: "No",
        7: "Refused",
        9: "Don't know",
    },
}

# Categorical columns to label
CATEGORICALS = [
    "dietary_recall_status",
    "breast_fed_infant_either_day",
    "num_days_of_intake",
    "intake_day_of_week",
    "respondent_language",
    "combination_food_type",
    "eating_occasion_name",
    "food_source",
    "ate_meal_at_home",
]

# ---------- Helper: map codes -> "Label [count]" safely ----------
def with_counts(series: pd.Series, base_map: dict, missing_label: str = "Missing") -> pd.Series:
    """
    - Treats any '.' as NaN (already handled at read).
    - Coerces numeric-like to numeric so int keys match.
    - Unknown non-numeric codes fall back to 'raw_value [count]'.
    - True missing -> 'Missing [count]'.
    """
    s_num = pd.to_numeric(series, errors="coerce")
    vc_num = s_num.value_counts(dropna=False)

    code_to_label_count = {
        code: f"{label} [{int(vc_num.get(code, 0))}]"
        for code, label in base_map.items()
    }

    out = s_num.map(code_to_label_count)

    # Non-numeric but present -> keep raw with counts
    mask_non_numeric = s_num.isna() & series.notna()
    if mask_non_numeric.any():
        raw = series[mask_non_numeric]
        raw_vc = raw.value_counts(dropna=False)
        fallback = {k: f"{k} [{int(raw_vc.get(k, 0))}]" for k in raw_vc.index}
        out.loc[mask_non_numeric] = raw.map(fallback)

    miss_ct = int(vc_num.get(np.nan, 0))
    out = out.fillna(f"{missing_label} [{miss_ct}]")
    return out.astype("string")

# ---------- Build labeled copy ----------
df_lab = df.copy()

# Label WTDR2D zeros as "Day 2 dietary recall not done/incomplete [count]"; keep positives numeric
if "wt_diet_day2" in df_lab.columns:
    s = pd.to_numeric(df_lab["wt_diet_day2"], errors="coerce")
    is_zero = s.fillna(0.0).eq(0.0)
    zero_ct = int(is_zero.sum())
    label = f"Day 2 dietary recall not done/incomplete [{zero_ct}]"
    df_lab["wt_diet_day2"] = np.where(is_zero, label, s)

# Categorical mappings with counts
for col in CATEGORICALS:
    if col in df_lab.columns:
        df_lab[col] = with_counts(df_lab[col], LBL[col], missing_label="Missing")

# Leave these numeric/ID/time fields numeric
NUMERIC_KEEP = [
    "respondent_id",
    "wt_diet_day1",                      # keep numeric
    "food_item_line_number",
    "interviewer_id_code",
    "days_between_intake_and_hh_interview",
    "combination_food_number",
    "time_of_eating_occasion_hhmm",
    "usda_food_code",
    "grams","energy_kcal","protein_g","carbohydrate_g","sugars_total_g",
    "fiber_dietary_g","fat_total_g","fat_saturated_g","fat_monounsaturated_g",
    "fat_polyunsaturated_g","cholesterol_mg","vitamin_e_alpha_tocopherol_mg",
    "vitamin_e_alpha_tocopherol_added_mg","retinol_mcg","vitamin_a_rae_mcg",
    "alpha_carotene_mcg","beta_carotene_mcg","beta_cryptoxanthin_mcg",
    "lycopene_mcg","lutein_zeaxanthin_mcg","thiamin_b1_mg","riboflavin_b2_mg",
    "niacin_mg","vitamin_b6_mg","folate_total_mcg","folic_acid_mcg",
    "food_folate_mcg","folate_dfe_mcg","choline_total_mg","vitamin_b12_mcg",
    "vitamin_b12_added_mcg","vitamin_c_mg","vitamin_d_d2_d3_mcg","vitamin_k_mcg",
    "calcium_mg","phosphorus_mg","magnesium_mg","iron_mg","zinc_mg","copper_mg",
    "sodium_mg","potassium_mg","selenium_mcg","caffeine_mg","theobromine_mg",
    "alcohol_g","moisture_g","sfa_4_0_butanoic_g","sfa_6_0_hexanoic_g",
    "sfa_8_0_octanoic_g","sfa_10_0_decanoic_g","sfa_12_0_dodecanoic_g",
    "sfa_14_0_tetradecanoic_g","sfa_16_0_hexadecanoic_g","sfa_18_0_octadecanoic_g",
    "mfa_16_1_hexadecenoic_g","mfa_18_1_octadecenoic_g","mfa_20_1_eicosenoic_g",
    "mfa_22_1_docosenoic_g","pfa_18_2_octadecadienoic_g","pfa_18_3_octadecatrienoic_g",
    "pfa_18_4_octadecatetraenoic_g","pfa_20_4_eicosatetraenoic_g",
    "pfa_20_5_eicosapentaenoic_g","pfa_22_5_docosapentaenoic_g","pfa_22_6_docosahexaenoic_g",
]
for col in NUMERIC_KEEP:
    if col in df_lab.columns:
        df_lab[col] = pd.to_numeric(df_lab[col], errors="coerce")

# ---------- Write labeled ----------
df_lab.to_csv(labeled_csv, index=False)
print(f"Saved labeled CSV to: {labeled_csv}")

Saved renamed CSV to: /content/drive/MyDrive/MSML610 Project/CSV_meaningful/DR2IFF_L_meaningful.csv
Saved labeled CSV to: /content/drive/MyDrive/MSML610 Project/CSV_meaningful/DR2IFF_L_meaningful_labeled.csv


In [20]:
import os
import numpy as np
import pandas as pd

# ---------- Paths ----------
in_csv  = "/content/drive/MyDrive/MSML610 Project/CSV_meaningful/DSQIDS_L_meaningful.csv"
out_dir = "/content/drive/MyDrive/MSML610 Project/CSV_meaningful"
os.makedirs(out_dir, exist_ok=True)
labeled_csv = os.path.join(out_dir, "DSQIDS_L_meaningful_labeled.csv")

# ---------- Read (treat '.' as missing just in case) ----------
df = pd.read_csv(in_csv, na_values=["."], keep_default_na=True)

# ---------- Label dictionaries (codes -> text) ----------
LBL = {
    "antacid_reported_as_supplement": {
        0: "Non-antacid supplement",
        1: "Antacid reported with dietary supplement (DSQ)",
        2: "Antacid reported with medication (RXQ)",
    },
    "container_seen": {
        1: "Yes",
        2: "No",
    },
    "matching_code": {
        1: "Exact or near exact match",
        2: "Probable match",
        3: "Generic match",
        4: "Reasonable match",
        5: "Default match",
        6: "No match",
        7: "Refused",
        9: "Don't know",
    },
    "dosage_form": {
        1: "TABLETS/CAPSULES/PILLS/CAPLETS/SOFTGELS/GEL CAPS/VEGICAPS/CHEWABLE TABLETS",
        2: "Droppers",
        3: "Drops",
        5: "Injections/shots",
        6: "LOZENGES/COUGH DROPS",
        7: "Milliliters",
        11: "Tablespoons",
        12: "Teaspoons",
        13: "Wafers",
        15: "Cans",
        16: "Grams",
        17: "Dots",
        18: "Cups",
        19: "Sprays/Squirts",
        20: "CHEWS/GUMMIES",
        21: "Scoop",
        22: "CC",
        23: "Capful",
        27: "Ounces",
        28: "Packages/Packets",
        29: "Vial",
        30: "Gumball",
        31: "Strips",
        32: "Bottle",
        41: "Pieces",
        77: "Refused",
        99: "Don't know",
    },
}

# Categorical columns to fully label with counts
CATEGORICALS = [
    "antacid_reported_as_supplement",  # DSDANTA
    "container_seen",                  # DSD070
    "matching_code",                   # DSDMTCH
    "dosage_form",                     # DSD122U
]

# Numeric/ID fields we keep numeric
NUMERIC_KEEP = [
    "respondent_id",
    "supplement_id",
    "wt_diet_day1",                    # special handling below for zeros
    "days_taken_past_30",              # DSD103 (1..30); specials handled separately
    "quantity_taken_daily",            # DSD122Q (real); specials handled separately
    "reported_over_label_serving_size",

    # nutrient totals
    "energy_kcal","protein_g","carbohydrate_g","sugars_total_g","fiber_dietary_g",
    "fat_total_g","fat_saturated_g","fat_monounsaturated_g","fat_polyunsaturated_g",
    "cholesterol_mg","lycopene_mcg","lutein_zeaxanthin_mcg","thiamin_b1_mg",
    "riboflavin_b2_mg","niacin_mg","vitamin_b6_mg","folic_acid_mcg","folate_dfe_mcg",
    "choline_total_mg","vitamin_b12_mcg","vitamin_c_mg","vitamin_k_mcg",
    "vitamin_d_d2_d3_mcg","calcium_mg","phosphorus_mg","magnesium_mg","iron_mg",
    "zinc_mg","copper_mg","sodium_mg","potassium_mg","selenium_mcg","caffeine_mg",
    "iodine_mcg",
]

# ---------- Helpers ----------
def with_counts(series: pd.Series, base_map: dict, missing_label: str = "Missing") -> pd.Series:
    """
    Map codes -> 'Label [count]' using counts computed on the series.
    - Coerces numeric-like strings to numeric so int keys match.
    - Unknown non-numeric tokens fall back to 'raw_value [count]'.
    - True missing -> 'Missing [count]'.
    """
    s_num = pd.to_numeric(series, errors="coerce")
    vc_num = s_num.value_counts(dropna=False)

    code_to_label_count = {
        code: f"{label} [{int(vc_num.get(code, 0))}]"
        for code, label in base_map.items()
    }

    out = s_num.map(code_to_label_count)

    # Non-numeric but present -> keep raw with counts
    mask_non_numeric = s_num.isna() & series.notna()
    if mask_non_numeric.any():
        raw = series[mask_non_numeric]
        raw_vc = raw.value_counts(dropna=False)
        fallback = {k: f"{k} [{int(raw_vc.get(k, 0))}]" for k in raw_vc.index}
        out.loc[mask_non_numeric] = raw.map(fallback)

    miss_ct = int(vc_num.get(np.nan, 0))
    out = out.fillna(f"{missing_label} [{miss_ct}]")
    return out.astype("string")

def map_specials_with_counts(series: pd.Series, specials: dict, missing_label="Missing") -> pd.Series:
    """
    For fields that are mostly numeric ranges but have special codes:
    - Keep valid numeric values as numbers
    - Map special codes -> 'Label [count]'
    """
    # Work on a copy
    s_raw = series.copy()

    # Coerce to numeric, preserve originals for non-numeric specials
    s_num = pd.to_numeric(s_raw, errors="coerce")

    # Count specials (numeric-coded)
    vc_num = s_num.value_counts(dropna=False)
    specials_labels = {}
    for code, label in specials.items():
        # specials might be big integers; coerce to numeric for counting
        code_num = pd.to_numeric(pd.Series([code]), errors="coerce").iloc[0]
        ct = int(vc_num.get(code_num, 0))
        specials_labels[code_num] = f"{label} [{ct}]"

    out = s_num.map(specials_labels)

    # Where we mapped a special, keep the label; otherwise keep numeric
    mask_special = s_num.isin(list(specials_labels.keys()))
    out = out.where(mask_special, s_num)

    # Missing -> 'Missing [count]'
    miss_ct = int(vc_num.get(np.nan, 0))
    out = out.where(~s_num.isna(), f"{missing_label} [{miss_ct}]")

    return out

# ---------- Build labeled copy ----------
df_lab = df.copy()

# 1) Handle WTDRD1 zeros -> label, positives numeric
if "wt_diet_day1" in df_lab.columns:
    s = pd.to_numeric(df_lab["wt_diet_day1"], errors="coerce")
    is_zero = s.fillna(0.0).eq(0.0)
    zero_ct = int(is_zero.sum())
    zero_label = f"Day 1 dietary recall not done/incomplete [{zero_ct}]"
    df_lab["wt_diet_day1"] = np.where(is_zero, zero_label, s)

# 2) Fully labeled categoricals with counts
for col in CATEGORICALS:
    if col in df_lab.columns:
        df_lab[col] = with_counts(df_lab[col], LBL[col], missing_label="Missing")

# 3) Range variables with special codes: DSD103, DSD122Q
#    DSD103: 1..30 (numeric), 7777 Refused, 9999 Don't know
if "days_taken_past_30" in df_lab.columns:
    specials_103 = {7777: "Refused", 9999: "Don't know"}
    df_lab["days_taken_past_30"] = map_specials_with_counts(df_lab["days_taken_past_30"], specials_103)

#    DSD122Q: real value; specials 777777 Refused, 999999 Don't know
if "quantity_taken_daily" in df_lab.columns:
    specials_122q = {777777: "Refused", 999999: "Don't know"}
    df_lab["quantity_taken_daily"] = map_specials_with_counts(df_lab["quantity_taken_daily"], specials_122q)

# 4) Keep other numeric/ID fields numeric (coerce; respects NaN from '.')
for col in NUMERIC_KEEP:
    if col in df_lab.columns:
        df_lab[col] = pd.to_numeric(df_lab[col], errors="coerce")

# ---------- Save ----------
df_lab.to_csv(labeled_csv, index=False)
print(f"Saved labeled CSV to: {labeled_csv}")

Saved labeled CSV to: /content/drive/MyDrive/MSML610 Project/CSV_meaningful/DSQIDS_L_meaningful_labeled.csv


In [21]:
import os
import pandas as pd

# ---------- Paths ----------
in_csv  = "/content/drive/MyDrive/MSML610 Project/CSV/DSQTOT_L.csv"
out_dir = "/content/drive/MyDrive/MSML610 Project/CSV_meaningful"
os.makedirs(out_dir, exist_ok=True)
renamed_csv = os.path.join(out_dir, "DSQTOT_L_meaningful.csv")

# ---------- Read (treat "." as missing) ----------
df = pd.read_csv(in_csv, na_values=["."], keep_default_na=True)

# ---------- Map: NHANES code -> meaningful snake_case name ----------
CODE_TO_NAME = {
    "SEQN":       "respondent_id",
    "WTDRD1":     "wt_diet_day1",

    "DSDCOUNT":   "total_dietary_supplements_count",
    "DSDANCNT":   "total_antacids_count",
    "DSD010":     "any_dietary_supplements",
    "DSD010AN":   "any_antacids",

    "DSQTKCAL":   "energy_kcal",
    "DSQTPROT":   "protein_g",
    "DSQTCARB":   "carbohydrate_g",
    "DSQTSUGR":   "sugars_total_g",
    "DSQTFIBE":   "fiber_dietary_g",
    "DSQTTFAT":   "fat_total_g",
    "DSQTSFAT":   "fat_saturated_g",
    "DSQTMFAT":   "fat_monounsaturated_g",
    "DSQTPFAT":   "fat_polyunsaturated_g",
    "DSQTCHOL":   "cholesterol_mg",
    "DSQTLYCO":   "lycopene_mcg",
    "DSQTLZ":     "lutein_zeaxanthin_mcg",
    "DSQTVB1":    "thiamin_b1_mg",
    "DSQTVB2":    "riboflavin_b2_mg",
    "DSQTNIAC":   "niacin_mg",
    "DSQTVB6":    "vitamin_b6_mg",
    "DSQTFA":     "folic_acid_mcg",
    "DSQTFDFE":   "folate_dfe_mcg",
    "DSQTCHL":    "choline_total_mg",
    "DSQTVB12":   "vitamin_b12_mcg",
    "DSQTVC":     "vitamin_c_mg",
    "DSQTVK":     "vitamin_k_mcg",
    "DSQTVD":     "vitamin_d_d2_d3_mcg",
    "DSQTCALC":   "calcium_mg",
    "DSQTPHOS":   "phosphorus_mg",
    "DSQTMAGN":   "magnesium_mg",
    "DSQTIRON":   "iron_mg",
    "DSQTZINC":   "zinc_mg",
    "DSQTCOPP":   "copper_mg",
    "DSQTSODI":   "sodium_mg",
    "DSQTPOTA":   "potassium_mg",
    "DSQTSELE":   "selenium_mcg",
    "DSQTCAFF":   "caffeine_mg",
    "DSQTIODI":   "iodine_mcg",
}

# ---------- Rename ----------
present_to_name = {c: CODE_TO_NAME[c] for c in df.columns if c in CODE_TO_NAME}
missing = [k for k in CODE_TO_NAME if k not in df.columns]
if missing:
    print("Note: not found in header (not renamed):", ", ".join(missing))

df = df.rename(columns=present_to_name)

# ---------- Write ----------
df.to_csv(renamed_csv, index=False)
print(f"Saved renamed CSV to: {renamed_csv}")

Saved renamed CSV to: /content/drive/MyDrive/MSML610 Project/CSV_meaningful/DSQTOT_L_meaningful.csv


In [22]:
import os
import numpy as np
import pandas as pd

# ---------- Paths ----------
in_csv  = "/content/drive/MyDrive/MSML610 Project/CSV_meaningful/DSQTOT_L_meaningful.csv"
out_dir = "/content/drive/MyDrive/MSML610 Project/CSV_meaningful"
os.makedirs(out_dir, exist_ok=True)
labeled_csv = os.path.join(out_dir, "DSQTOT_L_meaningful_labeled.csv")

# ---------- Read (treat "." as missing just in case) ----------
df = pd.read_csv(in_csv, na_values=["."], keep_default_na=True)

# ---------- Label dictionaries ----------
LBL = {
    "any_dietary_supplements": {
        1: "Yes",
        2: "No",
        7: "Refused",
        9: "Don't know",
    },
    "any_antacids": {
        1: "Yes",
        2: "No",
        7: "Refused",
        9: "Don't know",
    },
}

# Range-with-specials maps
SPECIALS_DSDCOUNT = {77: "Refused", 99: "Don't know"}   # 0..30 else
SPECIALS_DSDANCNT = {77: "Refused", 99: "Don't know"}   # 0..3  else

# Numeric fields to keep numeric (coerce)
NUMERIC_KEEP = [
    "respondent_id",
    "total_dietary_supplements_count",  # mapped below for specials; keep numeric otherwise
    "total_antacids_count",             # mapped below for specials; keep numeric otherwise
    "wt_diet_day1",                     # special zero handling below

    # nutrients
    "energy_kcal","protein_g","carbohydrate_g","sugars_total_g","fiber_dietary_g",
    "fat_total_g","fat_saturated_g","fat_monounsaturated_g","fat_polyunsaturated_g",
    "cholesterol_mg","lycopene_mcg","lutein_zeaxanthin_mcg","thiamin_b1_mg",
    "riboflavin_b2_mg","niacin_mg","vitamin_b6_mg","folic_acid_mcg","folate_dfe_mcg",
    "choline_total_mg","vitamin_b12_mcg","vitamin_c_mg","vitamin_k_mcg",
    "vitamin_d_d2_d3_mcg","calcium_mg","phosphorus_mg","magnesium_mg","iron_mg",
    "zinc_mg","copper_mg","sodium_mg","potassium_mg","selenium_mcg","caffeine_mg",
    "iodine_mcg",
]

# ---------- Helpers ----------
def with_counts(series: pd.Series, base_map: dict, missing_label: str = "Missing") -> pd.Series:
    """
    Map integer codes -> 'Label [count]'.
    - Coerces numeric-like strings to numeric so int keys match.
    - Unknown non-numeric tokens fall back to 'raw_value [count]'.
    - True missing -> 'Missing [count]'.
    """
    s_num = pd.to_numeric(series, errors="coerce")
    vc_num = s_num.value_counts(dropna=False)

    code_to_label_count = {
        code: f"{label} [{int(vc_num.get(code, 0))}]"
        for code, label in base_map.items()
    }
    out = s_num.map(code_to_label_count)

    mask_non_numeric = s_num.isna() & series.notna()
    if mask_non_numeric.any():
        raw = series[mask_non_numeric]
        raw_vc = raw.value_counts(dropna=False)
        fallback = {k: f"{k} [{int(raw_vc.get(k, 0))}]" for k in raw_vc.index}
        out.loc[mask_non_numeric] = raw.map(fallback)

    miss_ct = int(vc_num.get(np.nan, 0))
    out = out.fillna(f"{missing_label} [{miss_ct}]")
    return out.astype("string")

def map_range_with_specials(series: pd.Series, specials: dict, missing_label="Missing") -> pd.Series:
    """
    For mostly numeric ranges with special codes:
    - Keep normal numeric values as numbers.
    - Map special codes -> 'Label [count]'.
    - Missing -> 'Missing [count]'.
    """
    s_num = pd.to_numeric(series, errors="coerce")
    vc_num = s_num.value_counts(dropna=False)

    specials_labels = {}
    for code, label in specials.items():
        code_num = pd.to_numeric(pd.Series([code]), errors="coerce").iloc[0]
        specials_labels[code_num] = f"{label} [{int(vc_num.get(code_num, 0))}]"

    out = s_num.map(specials_labels)
    is_special = s_num.isin(list(specials_labels.keys()))
    out = out.where(is_special, s_num)

    miss_ct = int(vc_num.get(np.nan, 0))
    out = out.where(~s_num.isna(), f"{missing_label} [{miss_ct}]")
    return out

# ---------- Build labeled copy ----------
df_lab = df.copy()

# 1) WTDRD1: label zeros as "Day 1 dietary recall not done/incomplete [count]"; keep positives numeric
if "wt_diet_day1" in df_lab.columns:
    s = pd.to_numeric(df_lab["wt_diet_day1"], errors="coerce")
    is_zero = s.fillna(0.0).eq(0.0)
    zero_ct = int(is_zero.sum())
    zero_label = f"Day 1 dietary recall not done/incomplete [{zero_ct}]"
    df_lab["wt_diet_day1"] = np.where(is_zero, zero_label, s)

# 2) Range variables with specials
if "total_dietary_supplements_count" in df_lab.columns:
    df_lab["total_dietary_supplements_count"] = map_range_with_specials(
        df_lab["total_dietary_supplements_count"], SPECIALS_DSDCOUNT
    )
if "total_antacids_count" in df_lab.columns:
    df_lab["total_antacids_count"] = map_range_with_specials(
        df_lab["total_antacids_count"], SPECIALS_DSDANCNT
    )

# 3) Fully coded categoricals
for col in ["any_dietary_supplements", "any_antacids"]:
    if col in df_lab.columns:
        df_lab[col] = with_counts(df_lab[col], LBL[col], missing_label="Missing")

# 4) Coerce remaining numeric/ID fields to numeric
for col in NUMERIC_KEEP:
    if col in df_lab.columns:
        df_lab[col] = pd.to_numeric(df_lab[col], errors="coerce")

# ---------- Save ----------
df_lab.to_csv(labeled_csv, index=False)
print(f"Saved labeled CSV to: {labeled_csv}")

Saved labeled CSV to: /content/drive/MyDrive/MSML610 Project/CSV_meaningful/DSQTOT_L_meaningful_labeled.csv


In [23]:
import os
import pandas as pd

# ---------- Paths ----------
in_csv  = "/content/drive/MyDrive/MSML610 Project/CSV/BPXO_L.csv"
out_dir = "/content/drive/MyDrive/MSML610 Project/CSV_meaningful"
os.makedirs(out_dir, exist_ok=True)
renamed_csv = os.path.join(out_dir, "BPXO_L_meaningful.csv")

# ---------- Read (treat "." as missing) ----------
df = pd.read_csv(in_csv, na_values=["."], keep_default_na=True)

# ---------- Map: NHANES code -> meaningful snake_case name ----------
CODE_TO_NAME = {
    "SEQN":     "respondent_id",
    "BPAOARM":  "arm_selected_oscillometric",
    "BPAOCSZ":  "cuff_size_code_oscillometric",
    "BPXOSY1":  "systolic_1st_osc",
    "BPXODI1":  "diastolic_1st_osc",
    "BPXOSY2":  "systolic_2nd_osc",
    "BPXODI2":  "diastolic_2nd_osc",
    "BPXOSY3":  "systolic_3rd_osc",
    "BPXODI3":  "diastolic_3rd_osc",
    "BPXOPLS1": "pulse_1st_osc",
    "BPXOPLS2": "pulse_2nd_osc",
    "BPXOPLS3": "pulse_3rd_osc",
}

# ---------- Rename ----------
present_to_name = {c: CODE_TO_NAME[c] for c in df.columns if c in CODE_TO_NAME}
missing = [k for k in CODE_TO_NAME if k not in df.columns]
if missing:
    print("Note: not found in header (not renamed):", ", ".join(missing))

df = df.rename(columns=present_to_name)

# ---------- Write ----------
df.to_csv(renamed_csv, index=False)
print(f"Saved renamed CSV to: {renamed_csv}")

Saved renamed CSV to: /content/drive/MyDrive/MSML610 Project/CSV_meaningful/BPXO_L_meaningful.csv


In [26]:
import os
import re
import numpy as np
import pandas as pd

# ---------- Paths ----------
in_csv  = "/content/drive/MyDrive/MSML610 Project/CSV/BPXO_L.csv"
out_dir = "/content/drive/MyDrive/MSML610 Project/CSV_meaningful"
os.makedirs(out_dir, exist_ok=True)
renamed_csv = os.path.join(out_dir, "BPXO_L_meaningful.csv")
labeled_csv = os.path.join(out_dir, "BPXO_L_meaningful_labeled.csv")

# ---------- Read (treat "." as missing) ----------
df = pd.read_csv(in_csv, na_values=["."], keep_default_na=True)

# ---------- Rename to meaningful headers ----------
CODE_TO_NAME = {
    "SEQN":     "respondent_id",
    "BPAOARM":  "arm_selected_oscillometric",
    "BPAOCSZ":  "cuff_size_code_oscillometric",
    "BPXOSY1":  "systolic_1st_osc",
    "BPXODI1":  "diastolic_1st_osc",
    "BPXOSY2":  "systolic_2nd_osc",
    "BPXODI2":  "diastolic_2nd_osc",
    "BPXOSY3":  "systolic_3rd_osc",
    "BPXODI3":  "diastolic_3rd_osc",
    "BPXOPLS1": "pulse_1st_osc",
    "BPXOPLS2": "pulse_2nd_osc",
    "BPXOPLS3": "pulse_3rd_osc",
}
present_to_name = {c: CODE_TO_NAME[c] for c in df.columns if c in CODE_TO_NAME}
missing = [k for k in CODE_TO_NAME if k not in df.columns]
if missing:
    print("Note: not found in header (not renamed):", ", ".join(missing))
df = df.rename(columns=present_to_name)

# ---------- Normalize junk text like b'R', B'R', bytes, whitespace ----------
def normalize_arm_code(val):
    """
    Convert arm codes to clean 'L'/'R':
      - decodes bytes
      - strips whitespace
      - removes byte-literal wrappers: b'R', B"R", etc.
      - uppercases
    Returns np.nan for empty/unknown.
    """
    if isinstance(val, (bytes, bytearray)):
        try:
            val = val.decode("utf-8", errors="ignore")
        except Exception:
            val = str(val)

    if pd.isna(val):
        return np.nan

    s = str(val).strip()

    # Remove Python byte-literal wrappers like b'R' or B"R"
    m = re.fullmatch(r"[bB][\"'](.*)[\"']", s)
    if m:
        s = m.group(1)

    s = s.strip().upper()
    if s in {"L", "R"}:
        return s
    # Sometimes CSV might contain quotes e.g., "'R'"
    s = s.strip("'\"")
    if s in {"L", "R"}:
        return s
    return np.nan  # anything else treated as missing

if "arm_selected_oscillometric" in df.columns:
    df["arm_selected_oscillometric"] = df["arm_selected_oscillometric"].map(normalize_arm_code)

# Cuff size should be numeric codes 2/3/4/5; coerce safely
if "cuff_size_code_oscillometric" in df.columns:
    df["cuff_size_code_oscillometric"] = pd.to_numeric(df["cuff_size_code_oscillometric"], errors="coerce")

# ---------- Write clean (meaningful) ----------
df.to_csv(renamed_csv, index=False)
print(f"Saved renamed CSV to: {renamed_csv}")

# ---------- Build labeled version with counts ----------
LBL = {
    "arm_selected_oscillometric": {
        "L": "Left",
        "R": "Right",
    },
    "cuff_size_code_oscillometric": {
        2: "17–21.9 cm (bladder 9.20×16.68 cm)",
        3: "22–31.9 cm (bladder 12.49×23.52 cm)",
        4: "32–41.9 cm (bladder 14.98×31.19 cm)",
        5: "42–50 cm (bladder 17.98×37.89 cm)",
    },
}

def with_counts(series: pd.Series, base_map: dict, missing_label: str = "Missing") -> pd.Series:
    """
    Map codes -> 'Label [count]'.
    Handles string-coded (L/R) and numeric-coded (2/3/4/5).
    Unknown tokens echoed with their own counts; true missing -> 'Missing [count]'.
    """
    s_raw = series.astype("object")
    s_num = pd.to_numeric(s_raw, errors="coerce")

    vc_raw = s_raw.value_counts(dropna=False)
    vc_num = s_num.value_counts(dropna=False)

    out = pd.Series(index=series.index, dtype="object")

    # string keys first
    raw_keys = {k: v for k, v in base_map.items() if isinstance(k, str)}
    if raw_keys:
        out = s_raw.map({k: f"{label} [{int(vc_raw.get(k, 0))}]" for k, label in raw_keys.items()})

    # numeric keys next
    num_keys = {k: v for k, v in base_map.items() if not isinstance(k, str)}
    if num_keys:
        mapped_num = s_num.map({k: f"{label} [{int(vc_num.get(k, 0))}]" for k, label in num_keys.items()})
        out = out.where(out.notna(), mapped_num)

    # Unmapped but not missing -> echo token with its count
    mask_unmapped = out.isna() & s_raw.notna()
    if mask_unmapped.any():
        sub = s_raw[mask_unmapped]
        sub_vc = sub.value_counts(dropna=False)
        fallback = {k: f"{k} [{int(sub_vc.get(k, 0))}]" for k in sub_vc.index}
        out.loc[mask_unmapped] = sub.map(fallback)

    # Missing -> 'Missing [count]'
    miss_ct = int(vc_raw.get(np.nan, 0))
    out = out.fillna(f"Missing [{miss_ct}]")
    return out.astype("string")

df_lab = df.copy()

# Apply label+count mapping (after normalization done above)
if "arm_selected_oscillometric" in df_lab.columns:
    df_lab["arm_selected_oscillometric"] = with_counts(
        df_lab["arm_selected_oscillometric"], LBL["arm_selected_oscillometric"]
    )
if "cuff_size_code_oscillometric" in df_lab.columns:
    df_lab["cuff_size_code_oscillometric"] = with_counts(
        df_lab["cuff_size_code_oscillometric"], LBL["cuff_size_code_oscillometric"]
    )

# Keep numeric readings numeric
NUMERIC_KEEP = [
    "respondent_id",
    "systolic_1st_osc","diastolic_1st_osc","systolic_2nd_osc","diastolic_2nd_osc",
    "systolic_3rd_osc","diastolic_3rd_osc",
    "pulse_1st_osc","pulse_2nd_osc","pulse_3rd_osc",
]
for col in NUMERIC_KEEP:
    if col in df_lab.columns:
        df_lab[col] = pd.to_numeric(df_lab[col], errors="coerce")

# ---------- Write labeled ----------
df_lab.to_csv(labeled_csv, index=False)
print(f"Saved labeled CSV to: {labeled_csv}")

Saved renamed CSV to: /content/drive/MyDrive/MSML610 Project/CSV_meaningful/BPXO_L_meaningful.csv
Saved labeled CSV to: /content/drive/MyDrive/MSML610 Project/CSV_meaningful/BPXO_L_meaningful_labeled.csv


In [27]:
import os
import pandas as pd

# ---------- Paths ----------
in_csv  = "/content/drive/MyDrive/MSML610 Project/CSV/BMX_L.csv"
out_dir = "/content/drive/MyDrive/MSML610 Project/CSV_meaningful"
os.makedirs(out_dir, exist_ok=True)
renamed_csv = os.path.join(out_dir, "BMX_L_meaningful.csv")

# ---------- Read (treat "." as missing) ----------
df = pd.read_csv(in_csv, na_values=["."], keep_default_na=True)

# ---------- Map: NHANES code -> meaningful snake_case name ----------
CODE_TO_NAME = {
    "SEQN":       "respondent_id",
    "BMDSTATS":   "component_status_code",

    "BMXWT":      "weight_kg",
    "BMIWT":      "weight_comment",

    "BMXRECUM":   "recumbent_length_cm",
    "BMIRECUM":   "recumbent_length_comment",

    "BMXHEAD":    "head_circumference_cm",
    "BMIHEAD":    "head_circumference_comment",

    "BMXHT":      "standing_height_cm",
    "BMIHT":      "standing_height_comment",

    "BMXBMI":     "bmi_kg_m2",
    "BMDBMIC":    "bmi_category_children_youth",

    "BMXLEG":     "upper_leg_length_cm",
    "BMILEG":     "upper_leg_length_comment",

    "BMXARML":    "upper_arm_length_cm",
    "BMIARML":    "upper_arm_length_comment",

    "BMXARMC":    "arm_circumference_cm",
    "BMIARMC":    "arm_circumference_comment",

    "BMXWAIST":   "waist_circumference_cm",
    "BMIWAIST":   "waist_circumference_comment",

    "BMXHIP":     "hip_circumference_cm",
    "BMIHIP":     "hip_circumference_comment",
}

# ---------- Rename ----------
present_to_name = {c: CODE_TO_NAME[c] for c in df.columns if c in CODE_TO_NAME}
missing = [k for k in CODE_TO_NAME if k not in df.columns]
if missing:
    print("Note: not found in header (not renamed):", ", ".join(missing))

df = df.rename(columns=present_to_name)

# ---------- Write ----------
df.to_csv(renamed_csv, index=False)
print(f"Saved renamed CSV to: {renamed_csv}")

Saved renamed CSV to: /content/drive/MyDrive/MSML610 Project/CSV_meaningful/BMX_L_meaningful.csv


In [28]:
import os
import numpy as np
import pandas as pd

# ---------- Paths ----------
in_csv  = "/content/drive/MyDrive/MSML610 Project/CSV_meaningful/BMX_L_meaningful.csv"
out_dir = "/content/drive/MyDrive/MSML610 Project/CSV_meaningful"
os.makedirs(out_dir, exist_ok=True)
labeled_csv = os.path.join(out_dir, "BMX_L_meaningful_labeled.csv")

# ---------- Read (treat '.' as missing just in case) ----------
df = pd.read_csv(in_csv, na_values=["."], keep_default_na=True)

# ---------- Label dictionaries from your codebook ----------
LBL = {
    "component_status_code": {          # BMDSTATS
        1: "Complete data for age group",
        2: "Partial: Only height and weight obtained",
        3: "Other partial exam",
        4: "No body measures exam data",
    },
    "weight_comment": {                 # BMIWT
        1: "Could not obtain",
        3: "Clothing",
        4: "Medical appliance",
    },
    "recumbent_length_comment": {       # BMIRECUM
        1: "Could not obtain",
        3: "Not straight",
    },
    "standing_height_comment": {        # BMIHT
        1: "Could not obtain",
        3: "Not straight",
    },
    "bmi_category_children_youth": {    # BMDBMIC
        1: "Underweight",
        2: "Normal weight",
        3: "Overweight",
        4: "Obese",
    },
    "upper_leg_length_comment": {       # BMILEG
        1: "Could not obtain",
    },
}

# Categorical columns to label (exactly the ones you supplied)
CATEGORICALS = [
    "component_status_code",
    "weight_comment",
    "recumbent_length_comment",
    "standing_height_comment",
    "bmi_category_children_youth",
    "upper_leg_length_comment",
]

# Numeric/ID fields to keep numeric
NUMERIC_KEEP = [
    "respondent_id",
    "weight_kg","recumbent_length_cm","head_circumference_cm","standing_height_cm",
    "bmi_kg_m2","upper_leg_length_cm","upper_arm_length_cm","arm_circumference_cm",
    "waist_circumference_cm","hip_circumference_cm",
    # if any other *_comment columns exist but not in LBL, keep them as-is
]

# ---------- Helpers ----------
def with_counts(series: pd.Series, base_map: dict, missing_label: str = "Missing") -> pd.Series:
    """
    Map integer codes -> 'Label [count]'.
    - Coerces numeric-like strings so int keys match.
    - Unknown non-missing tokens echoed as 'raw_value [count]'.
    - True missing -> 'Missing [count]'.
    """
    s_raw = series.astype("object")
    s_num = pd.to_numeric(s_raw, errors="coerce")

    vc_num = s_num.value_counts(dropna=False)
    vc_raw = s_raw.value_counts(dropna=False)

    # build mapping with counts for known codes
    mapped = s_num.map({code: f"{label} [{int(vc_num.get(code, 0))}]"
                        for code, label in base_map.items()})

    # unknown but not missing -> echo raw token with its count
    mask_unmapped = mapped.isna() & s_raw.notna()
    if mask_unmapped.any():
        sub = s_raw[mask_unmapped]
        sub_vc = sub.value_counts(dropna=False)
        fallback = {k: f"{k} [{int(sub_vc.get(k, 0))}]" for k in sub_vc.index}
        mapped.loc[mask_unmapped] = sub.map(fallback)

    # true missing -> Missing [count]
    miss_ct = int(vc_raw.get(np.nan, 0))
    mapped = mapped.fillna(f"{missing_label} [{miss_ct}]")
    return mapped.astype("string")

# ---------- Build labeled copy ----------
df_lab = df.copy()

# Apply label mappings
for col in CATEGORICALS:
    if col in df_lab.columns:
        df_lab[col] = with_counts(df_lab[col], LBL[col], missing_label="Missing")

# Keep measurements numeric
for col in NUMERIC_KEEP:
    if col in df_lab.columns:
        df_lab[col] = pd.to_numeric(df_lab[col], errors="coerce")

# ---------- Save ----------
df_lab.to_csv(labeled_csv, index=False)
print(f"Saved labeled CSV to: {labeled_csv}")

Saved labeled CSV to: /content/drive/MyDrive/MSML610 Project/CSV_meaningful/BMX_L_meaningful_labeled.csv


In [30]:
import os
import numpy as np
import pandas as pd

# ---------- Paths ----------
in_csv  = "/content/drive/MyDrive/MSML610 Project/CSV/TCHOL_L.csv"
out_dir = "/content/drive/MyDrive/MSML610 Project/CSV_meaningful"
os.makedirs(out_dir, exist_ok=True)
renamed_csv = os.path.join(out_dir, "TCHOL_L_meaningful.csv")
labeled_csv = os.path.join(out_dir, "TCHOL_L_meaningful_labeled.csv")

# ---------- Read source (treat "." as missing) ----------
df = pd.read_csv(in_csv, na_values=["."], keep_default_na=True)

# ---------- Rename to meaningful headers ----------
CODE_TO_NAME = {
    "SEQN":    "respondent_id",
    "WTPH2YR": "wt_phlebotomy_2yr",
    "LBXTC":   "total_cholesterol_mg_dl",
    "LBDTCSI": "total_cholesterol_mmol_l",
}
present_to_name = {c: CODE_TO_NAME[c] for c in df.columns if c in CODE_TO_NAME}
missing = [k for k in CODE_TO_NAME if k not in df.columns]
if missing:
    print("Note: not found in header (not renamed):", ", ".join(missing))
df = df.rename(columns=present_to_name)

# Coerce numeric for continuous/ID fields
for col in ["respondent_id", "wt_phlebotomy_2yr",
            "total_cholesterol_mg_dl", "total_cholesterol_mmol_l"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# ---------- Write the clean renamed file ----------
df.to_csv(renamed_csv, index=False)
print(f"Saved renamed CSV to: {renamed_csv}")

# ---------- Build labeled version ----------
df_lab = df.copy()

# WTPH2YR special: 0 -> "No blood sample provided [count]"; positives stay numeric
if "wt_phlebotomy_2yr" in df_lab.columns:
    s = pd.to_numeric(df_lab["wt_phlebotomy_2yr"], errors="coerce")
    is_zero = s.fillna(0.0).eq(0.0)
    zero_ct = int(is_zero.sum())
    zero_label = f"No blood sample provided [{zero_ct}]"
    df_lab["wt_phlebotomy_2yr"] = np.where(is_zero, zero_label, s)

# Keep cholesterol measures numeric (range values only, NaN for missing)
for col in ["total_cholesterol_mg_dl", "total_cholesterol_mmol_l", "respondent_id"]:
    if col in df_lab.columns:
        df_lab[col] = pd.to_numeric(df_lab[col], errors="coerce")

# ---------- Write labeled ----------
df_lab.to_csv(labeled_csv, index=False)
print(f"Saved labeled CSV to: {labeled_csv}")

Saved renamed CSV to: /content/drive/MyDrive/MSML610 Project/CSV_meaningful/TCHOL_L_meaningful.csv
Saved labeled CSV to: /content/drive/MyDrive/MSML610 Project/CSV_meaningful/TCHOL_L_meaningful_labeled.csv


In [32]:
import os
import numpy as np
import pandas as pd

# ---------- Paths ----------
in_csv  = "/content/drive/MyDrive/MSML610 Project/CSV/HDL_L.csv"
out_dir = "/content/drive/MyDrive/MSML610 Project/CSV_meaningful"
os.makedirs(out_dir, exist_ok=True)
renamed_csv = os.path.join(out_dir, "HDL_L_meaningful.csv")
labeled_csv = os.path.join(out_dir, "HDL_L_meaningful_labeled.csv")

# ---------- Read (treat "." as missing) ----------
df = pd.read_csv(in_csv, na_values=["."], keep_default_na=True)

# ---------- Rename to meaningful headers ----------
CODE_TO_NAME = {
    "SEQN":     "respondent_id",
    "WTPH2YR":  "wt_phlebotomy_2yr",
    "LBDHDD":   "hdl_cholesterol_mg_dl",
    "LBDHDDSI": "hdl_cholesterol_mmol_l",
}
present_to_name = {c: CODE_TO_NAME[c] for c in df.columns if c in CODE_TO_NAME}
missing = [k for k in CODE_TO_NAME if k not in df.columns]
if missing:
    print("Note: not found in header (not renamed):", ", ".join(missing))
df = df.rename(columns=present_to_name)

# ---------- Coerce numeric ----------
for col in ["respondent_id", "wt_phlebotomy_2yr",
            "hdl_cholesterol_mg_dl", "hdl_cholesterol_mmol_l"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# ---------- Write clean (meaningful) ----------
df.to_csv(renamed_csv, index=False)
print(f"Saved renamed CSV to: {renamed_csv}")

# ---------- Build labeled version ----------
df_lab = df.copy()

# WTPH2YR: 0 -> "No blood sample provided [count]"; positives stay numeric
if "wt_phlebotomy_2yr" in df_lab.columns:
    s = pd.to_numeric(df_lab["wt_phlebotomy_2yr"], errors="coerce")
    is_zero = s.fillna(0.0).eq(0.0)
    zero_ct = int(is_zero.sum())
    zero_label = f"No blood sample provided [{zero_ct}]"
    df_lab["wt_phlebotomy_2yr"] = np.where(is_zero, zero_label, s)

# Keep HDL measures numeric; NaN for missing
for col in ["hdl_cholesterol_mg_dl", "hdl_cholesterol_mmol_l", "respondent_id"]:
    if col in df_lab.columns:
        df_lab[col] = pd.to_numeric(df_lab[col], errors="coerce")

# ---------- Write labeled ----------
df_lab.to_csv(labeled_csv, index=False)
print(f"Saved labeled CSV to: {labeled_csv}")

Saved renamed CSV to: /content/drive/MyDrive/MSML610 Project/CSV_meaningful/HDL_L_meaningful.csv
Saved labeled CSV to: /content/drive/MyDrive/MSML610 Project/CSV_meaningful/HDL_L_meaningful_labeled.csv


In [33]:
import os
import numpy as np
import pandas as pd

# ---------- Paths ----------
in_csv  = "/content/drive/MyDrive/MSML610 Project/CSV/GLU_L.csv"
out_dir = "/content/drive/MyDrive/MSML610 Project/CSV_meaningful"
os.makedirs(out_dir, exist_ok=True)
renamed_csv = os.path.join(out_dir, "GLU_L_meaningful.csv")
labeled_csv = os.path.join(out_dir, "GLU_L_meaningful_labeled.csv")

# ---------- Read (treat "." as missing) ----------
df = pd.read_csv(in_csv, na_values=["."], keep_default_na=True)

# ---------- Rename to meaningful headers ----------
CODE_TO_NAME = {
    "SEQN":     "respondent_id",
    "WTSAF2YR": "wt_fasting_subsample_2yr_mec",
    "LBXGLU":   "fasting_glucose_mg_dl",
    "LBDGLUSI": "fasting_glucose_mmol_l",
}
present_to_name = {c: CODE_TO_NAME[c] for c in df.columns if c in CODE_TO_NAME}
missing = [k for k in CODE_TO_NAME if k not in df.columns]
if missing:
    print("Note: not found in header (not renamed):", ", ".join(missing))
df = df.rename(columns=present_to_name)

# ---------- Coerce numeric ----------
for col in ["respondent_id", "wt_fasting_subsample_2yr_mec",
            "fasting_glucose_mg_dl", "fasting_glucose_mmol_l"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# ---------- Write clean (meaningful) ----------
df.to_csv(renamed_csv, index=False)
print(f"Saved renamed CSV to: {renamed_csv}")

# ---------- Build labeled version ----------
df_lab = df.copy()

def map_range_with_specials(series: pd.Series, specials: dict, missing_label="Missing") -> pd.Series:
    """
    Keep normal numeric values as numbers; map special codes to 'Label [count]';
    true missing -> 'Missing [count]'.
    """
    s = pd.to_numeric(series, errors="coerce")
    vc = s.value_counts(dropna=False)

    sp_labels = {}
    for k, v in specials.items():
        k_num = pd.to_numeric(pd.Series([k]), errors="coerce").iloc[0]
        sp_labels[k_num] = f"{v} [{int(vc.get(k_num, 0))}]"

    out = s.map(sp_labels)
    is_special = s.isin(list(sp_labels.keys()))
    out = out.where(is_special, s)

    miss_ct = int(vc.get(np.nan, 0))
    out = out.where(~s.isna(), f"{missing_label} [{miss_ct}]")
    return out

# WTSAF2YR: 0 -> "No Lab Result or Not Fasting for 8 to <24 hours [count]"
if "wt_fasting_subsample_2yr_mec" in df_lab.columns:
    df_lab["wt_fasting_subsample_2yr_mec"] = map_range_with_specials(
        df_lab["wt_fasting_subsample_2yr_mec"],
        specials={0: "No Lab Result or Not Fasting for 8 to <24 hours"}
    )

# Keep glucose measures numeric; NaN for missing
for col in ["fasting_glucose_mg_dl", "fasting_glucose_mmol_l", "respondent_id"]:
    if col in df_lab.columns:
        df_lab[col] = pd.to_numeric(df_lab[col], errors="coerce")

# ---------- Write labeled ----------
df_lab.to_csv(labeled_csv, index=False)
print(f"Saved labeled CSV to: {labeled_csv}")


Saved renamed CSV to: /content/drive/MyDrive/MSML610 Project/CSV_meaningful/GLU_L_meaningful.csv
Saved labeled CSV to: /content/drive/MyDrive/MSML610 Project/CSV_meaningful/GLU_L_meaningful_labeled.csv


In [34]:
import os
import numpy as np
import pandas as pd

# ---------- Paths ----------
in_csv  = "/content/drive/MyDrive/MSML610 Project/CSV/TRIGLY_L.csv"
out_dir = "/content/drive/MyDrive/MSML610 Project/CSV_meaningful"
os.makedirs(out_dir, exist_ok=True)
renamed_csv = os.path.join(out_dir, "TRIGLY_L_meaningful.csv")
labeled_csv = os.path.join(out_dir, "TRIGLY_L_meaningful_labeled.csv")

# ---------- Read (treat "." as missing) ----------
df = pd.read_csv(in_csv, na_values=["."], keep_default_na=True)

# ---------- Rename to meaningful headers ----------
CODE_TO_NAME = {
    "SEQN":     "respondent_id",
    "WTSAF2YR": "wt_fasting_subsample_2yr_mec",

    "LBXTLG":   "triglyceride_mg_dl",
    "LBDTRSI":  "triglyceride_mmol_l",

    "LBDLDL":   "ldl_cholesterol_friedewald_mg_dl",
    "LBDLDLSI": "ldl_cholesterol_friedewald_mmol_l",

    "LBDLDLM":  "ldl_cholesterol_martin_hopkins_mg_dl",
    "LBDLDMSI": "ldl_cholesterol_martin_hopkins_mmol_l",

    "LBDLDLN":  "ldl_cholesterol_nih_eq2_mg_dl",
    "LBDLDNSI": "ldl_cholesterol_nih_eq2_mmol_l",
}
present_to_name = {c: CODE_TO_NAME[c] for c in df.columns if c in CODE_TO_NAME}
missing = [k for k in CODE_TO_NAME if k not in df.columns]
if missing:
    print("Note: not found in header (not renamed):", ", ".join(missing))
df = df.rename(columns=present_to_name)

# ---------- Coerce numeric for IDs/measures ----------
NUMERIC_COLS = [
    "respondent_id",
    "wt_fasting_subsample_2yr_mec",
    "triglyceride_mg_dl","triglyceride_mmol_l",
    "ldl_cholesterol_friedewald_mg_dl","ldl_cholesterol_friedewald_mmol_l",
    "ldl_cholesterol_martin_hopkins_mg_dl","ldl_cholesterol_martin_hopkins_mmol_l",
    "ldl_cholesterol_nih_eq2_mg_dl","ldl_cholesterol_nih_eq2_mmol_l",
]
for col in NUMERIC_COLS:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# ---------- Write clean (meaningful) ----------
df.to_csv(renamed_csv, index=False)
print(f"Saved renamed CSV to: {renamed_csv}")

# ---------- Build labeled version ----------
df_lab = df.copy()

def map_range_with_specials(series: pd.Series, specials: dict, missing_label="Missing") -> pd.Series:
    """
    Keep normal numeric values as numbers; map special codes to 'Label [count]';
    true missing -> 'Missing [count]'.
    """
    s = pd.to_numeric(series, errors="coerce")
    vc = s.value_counts(dropna=False)

    sp_labels = {}
    for k, v in specials.items():
        k_num = pd.to_numeric(pd.Series([k]), errors="coerce").iloc[0]
        sp_labels[k_num] = f"{v} [{int(vc.get(k_num, 0))}]"

    out = s.map(sp_labels)
    is_special = s.isin(list(sp_labels.keys()))
    out = out.where(is_special, s)

    miss_ct = int(vc.get(np.nan, 0))
    out = out.where(~s.isna(), f"{missing_label} [{miss_ct}]")
    return out

# WTSAF2YR: 0 -> "No Lab Result or Not Fasting for 8 to <24 hours [count]"
if "wt_fasting_subsample_2yr_mec" in df_lab.columns:
    df_lab["wt_fasting_subsample_2yr_mec"] = map_range_with_specials(
        df_lab["wt_fasting_subsample_2yr_mec"],
        specials={0: "No Lab Result or Not Fasting for 8 to <24 hours"}
    )

# Keep all lipid measures and respondent_id numeric in labeled file too
for col in NUMERIC_COLS:
    if col in df_lab.columns and col != "wt_fasting_subsample_2yr_mec":  # already handled with specials above
        df_lab[col] = pd.to_numeric(df_lab[col], errors="coerce")

# ---------- Write labeled ----------
df_lab.to_csv(labeled_csv, index=False)
print(f"Saved labeled CSV to: {labeled_csv}")

Saved renamed CSV to: /content/drive/MyDrive/MSML610 Project/CSV_meaningful/TRIGLY_L_meaningful.csv
Saved labeled CSV to: /content/drive/MyDrive/MSML610 Project/CSV_meaningful/TRIGLY_L_meaningful_labeled.csv


In [35]:
import os
import numpy as np
import pandas as pd

# ---------- Paths ----------
in_csv  = "/content/drive/MyDrive/MSML610 Project/CSV/HSCRP_L.csv"
out_dir = "/content/drive/MyDrive/MSML610 Project/CSV_meaningful"
os.makedirs(out_dir, exist_ok=True)
renamed_csv = os.path.join(out_dir, "HSCRP_L_meaningful.csv")
labeled_csv = os.path.join(out_dir, "HSCRP_L_meaningful_labeled.csv")

# ---------- Read (treat "." as missing) ----------
df = pd.read_csv(in_csv, na_values=["."], keep_default_na=True)

# ---------- Rename to meaningful headers ----------
CODE_TO_NAME = {
    "SEQN":     "respondent_id",
    "WTPH2YR":  "wt_phlebotomy_2yr",
    "LBXHSCRP": "hs_crp_mg_l",
    "LBDHRPLC": "hs_crp_comment_code",
}
present_to_name = {c: CODE_TO_NAME[c] for c in df.columns if c in CODE_TO_NAME}
missing = [k for k in CODE_TO_NAME if k not in df.columns]
if missing:
    print("Note: not found in header (not renamed):", ", ".join(missing))
df = df.rename(columns=present_to_name)

# ---------- Coerce numeric for ID/measures/codes ----------
for col in ["respondent_id", "wt_phlebotomy_2yr", "hs_crp_mg_l", "hs_crp_comment_code"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# ---------- Write clean (meaningful) ----------
df.to_csv(renamed_csv, index=False)
print(f"Saved renamed CSV to: {renamed_csv}")

# ---------- Build labeled version ----------
df_lab = df.copy()

def map_range_with_specials(series: pd.Series, specials: dict, missing_label="Missing") -> pd.Series:
    """
    Keep normal numeric values as numbers; map special codes to 'Label [count]';
    true missing -> 'Missing [count]'.
    """
    s = pd.to_numeric(series, errors="coerce")
    vc = s.value_counts(dropna=False)

    sp_labels = {}
    for k, v in specials.items():
        k_num = pd.to_numeric(pd.Series([k]), errors="coerce").iloc[0]
        sp_labels[k_num] = f"{v} [{int(vc.get(k_num, 0))}]"

    out = s.map(sp_labels)
    is_special = s.isin(list(sp_labels.keys()))
    out = out.where(is_special, s)

    miss_ct = int(vc.get(np.nan, 0))
    out = out.where(~s.isna(), f"{missing_label} [{miss_ct}]")
    return out

def label_with_counts(series: pd.Series, mapping: dict, missing_label="Missing") -> pd.Series:
    """
    Map integer codes -> 'Label [count]'; unknown non-missing echoed as 'raw [count]';
    true missing -> 'Missing [count]'.
    """
    s = pd.to_numeric(series, errors="coerce")
    vc = s.value_counts(dropna=False)
    labeled = s.map({k: f"{v} [{int(vc.get(k, 0))}]" for k, v in mapping.items()})

    # Unknown but non-missing -> echo with count
    mask_unmapped = labeled.isna() & s.notna()
    if mask_unmapped.any():
        sub = s[mask_unmapped]
        sub_vc = sub.value_counts(dropna=False)
        fallback = {k: f"{k} [{int(sub_vc.get(k, 0))}]" for k in sub_vc.index}
        labeled.loc[mask_unmapped] = sub.map(fallback)

    miss_ct = int(vc.get(np.nan, 0))
    labeled = labeled.fillna(f"{missing_label} [{miss_ct}]")
    return labeled.astype("string")

# WTPH2YR: 0 -> "No blood sample provided [count]"; positives stay numeric
if "wt_phlebotomy_2yr" in df_lab.columns:
    df_lab["wt_phlebotomy_2yr"] = map_range_with_specials(
        df_lab["wt_phlebotomy_2yr"],
        specials={0: "No blood sample provided"}
    )

# LBXHSCRP: keep numeric ranges; NaN for missing
if "hs_crp_mg_l" in df_lab.columns:
    df_lab["hs_crp_mg_l"] = pd.to_numeric(df_lab["hs_crp_mg_l"], errors="coerce")

# LBDHRPLC: 0 = At/above detection limit, 1 = Below lower detection limit
if "hs_crp_comment_code" in df_lab.columns:
    df_lab["hs_crp_comment_code"] = label_with_counts(
        df_lab["hs_crp_comment_code"],
        mapping={0: "At or above detection limit", 1: "Below lower detection limit"}
    )

# Keep respondent_id numeric in labeled file too
if "respondent_id" in df_lab.columns:
    df_lab["respondent_id"] = pd.to_numeric(df_lab["respondent_id"], errors="coerce")

# ---------- Write labeled ----------
df_lab.to_csv(labeled_csv, index=False)
print(f"Saved labeled CSV to: {labeled_csv}")

Saved renamed CSV to: /content/drive/MyDrive/MSML610 Project/CSV_meaningful/HSCRP_L_meaningful.csv
Saved labeled CSV to: /content/drive/MyDrive/MSML610 Project/CSV_meaningful/HSCRP_L_meaningful_labeled.csv
